In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sys, os
# This is not super pretty, but I think this is the best way to import stuff from ../data_generation_pipeline?
CODE_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../data_generation_pipeline"))
if CODE_ROOT not in sys.path:
    sys.path.insert(1, CODE_ROOT)

from spectra_stitching import load_forest_spectra, hash_spectra_args, find_closest_index
from compressed_spectra import load_spectra_cache
from create_training_dataset import add_noise_to_spectrum

In [ ]:
out_file_path = "/pfs/10/project/bw21g005/ly_alpha_sbi_paper/SDSS_spectra/SDSS_support_files/spectra_cache.npz"

data = load_spectra_cache(out_file_path)

In [ ]:
data["sigma_F"].shape

In [ ]:
wavelength_range = (3800, 4500)
n_specs = 10000
redshifts_to_use = [2.0, 2.1, 2.2, 2.3, 2.4, 2.5, 2.6, 2.7, 2.8, 2.9, 3.0]

args_hash = hash_spectra_args(wavelength_range, redshifts_to_use, None, n_specs, 1215.67, 123)

sim_spec_path = "/pfs/10/project/bw21g005/ly_alpha_sbi_paper/L50n512_suite/gridpoint0/lya_forest_spectra/" + f"forest_spectra_{args_hash}.hdf5"

print(sim_spec_path)

In [ ]:
wave_sim, flux_sim, usage_records = load_forest_spectra(sim_spec_path)

In [ ]:
# enforce redshift of quasar to be greater than right edge of interval

min_req_z = 4500/1215.67 - 1
print(min_req_z)

mask = (data["redshift"] >= min_req_z) & (data["valid_spectrum"] == 1)

wave_sdss = data["wavelength"]
flux_sdss = data["flux"][mask, :]
sigma_F_sdss = data["sigma_F"][mask, :]
mask_sdss = data["mask"][mask, :]
redshift_sdss = data["redshift"][mask]

print(flux_sdss.shape, sigma_F_sdss.shape, mask_sdss.shape)

In [ ]:
# cut spectra to interval of simulated spectra

i0 = find_closest_index(wave_sdss, wave_sim[0])
i1 = find_closest_index(wave_sdss, wave_sim[-1])

diff = abs(wave_sdss[i0:i1].shape[0] - wave_sim.shape[0])

if 1 >= diff > 0:
    i1 += diff
else:
    raise ValueError(f"Length mismatch between SDSS and simulation spectra: {wave_sdss[i0:i1].shape[0]} vs {wave_sim.shape[0]}")


print(sigma_F_sdss.shape)
wave_sdss = wave_sdss[i0:i1]
sigma_F_sdss = sigma_F_sdss[:, i0:i1]
mask_sdss = mask_sdss[:, i0:i1]
flux_sdss = flux_sdss[:, i0:i1]
print(sigma_F_sdss.shape)

In [ ]:
# enforce min of min_valid_pixels

min_valid_pixels = 100

mask_valid_pixels = np.sum(mask_sdss, axis=1) >= min_valid_pixels

wave_sdss = wave_sdss
sigma_F_sdss = sigma_F_sdss[mask_valid_pixels, :]
mask_sdss = mask_sdss[mask_valid_pixels, :]
redshift_sdss = redshift_sdss[mask_valid_pixels]
flux_sdss = flux_sdss[mask_valid_pixels, :]
print(sigma_F_sdss.shape)

In [ ]:
# Coverage map coloured by redshift: rows sorted by redshift, masked pixels painted black.
order = np.argsort(redshift_sdss)
z_sorted = redshift_sdss[order]
mask_sorted = mask_sdss[order]

# Good pixels carry their spectrum's redshift, masked pixels are NaN -> drawn black
z_image = np.where(mask_sorted, z_sorted[:, None], np.nan)

cmap = plt.get_cmap("jet").copy()
cmap.set_bad("black")  # mask_sdss == 0

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(
    z_image,
    cmap=cmap,
    vmin=z_sorted.min(),
    vmax=z_sorted.max(),
    origin="lower",
    aspect="auto",
    interpolation="nearest",
    extent=[wave_sdss[0], wave_sdss[-1], 0, z_image.shape[0]],
)
# Observed Lyman-beta wavelength for each spectrum's redshift
LYB_REST = 1025.7223  # Angstrom
lyb_obs = LYB_REST * (1.0 + z_sorted)
ax.plot(lyb_obs, np.arange(z_image.shape[0]), color="red", lw=1.0, label=r"Ly$\beta$")
ax.set_xlim(wave_sdss[0], wave_sdss[-1])
ax.legend(loc="upper left", labelcolor="white")

ax.set_xlabel("Wavelength [$\\AA$]")
ax.set_ylabel("# of Spectrum")
fig.colorbar(im, ax=ax, label="Redshift")

plt.show()

In [ ]:
# The cache stores the per-pixel error sigma_F, so build the S/N ourselves.
# Masked pixels are NaN in both flux and sigma_F, so they stay NaN here.
snr_sdss = flux_sdss / sigma_F_sdss

snr_mean_wave = np.nanmean(snr_sdss, axis=0)
snr_std_wave = np.nanstd(snr_sdss, axis=0)
snr_mean_spec = np.nanmean(snr_sdss, axis=1)

snr_median_wave = np.nanmedian(snr_sdss, axis=0)

snr_range = (0, 10)

# layout="none" disables the constrained/tight layout engine, which would
# otherwise ignore wspace and keep a gap between the two panels
fig, (ax, ax_hist) = plt.subplots(
    1, 2, figsize=(10, 4), sharey=True, layout="none",
    gridspec_kw={"width_ratios": [3, 1]},
)

ax.plot(wave_sdss, snr_mean_wave, color="green", label="Mean")
ax.fill_between(
    wave_sdss,
    snr_mean_wave - snr_std_wave,
    snr_mean_wave + snr_std_wave,
    color="green",
    alpha=0.3,
    label=r"1 $\sigma$"
)
ax.plot(wave_sdss, snr_median_wave, color="green", linestyle=":", label="Median", alpha=0.6)
ax.set_xlim(wave_sdss[0], wave_sdss[-1])
ax.set_ylim(*snr_range)
ax.set_xlabel("Wavelength [$\\AA$]")
ax.set_ylabel("SNR")
ax.legend(fontsize=10, loc="upper left")

ax_hist.hist(
    snr_mean_spec,
    bins=25,
    range=snr_range,
    orientation="horizontal",
    color="green",
    alpha=0.7,
    label=f"Mean per Spectrum\n(Total: {len(snr_mean_spec)})"
)
ax_hist.set_xlabel("# of Spectra")
ax_hist.tick_params(axis="y", labelleft=False, left=False)
ax_hist.xaxis.set_major_locator(plt.MaxNLocator(3, prune="lower"))
ax_hist.legend(fontsize=10, loc="upper right")

fig.tight_layout()             # fit the labels inside the figure
fig.subplots_adjust(wspace=0)  # then close the seam between the panels
plt.savefig("plots/snr_mean_hist.pdf", format="pdf")
plt.show()

In [ ]:
# draw one SDSS noise realisation (sigma_F + mask) per simulated spectrum

rng = np.random.default_rng(42)

n_sdss, n_sim = sigma_F_sdss.shape[0], flux_sim.shape[0]
replace = n_sdss < n_sim  # only reuse SDSS spectra if there are not enough of them

sample_idx = rng.choice(n_sdss, size=n_sim, replace=replace)

sigma_F_appl = sigma_F_sdss[sample_idx]
mask_appl = mask_sdss[sample_idx]
flux_sdss_appl = flux_sdss[sample_idx]

assert sigma_F_appl.shape == mask_appl.shape == flux_sim.shape, \
    f"shapes {sigma_F_appl.shape=} / {mask_appl.shape=} dont match {flux_sim.shape=}"

noisy_flux_sim = add_noise_to_spectrum(flux_sim, sigma_F_appl, mask_appl, rng=rng)

print(f"{flux_sim.shape} -> {noisy_flux_sim.shape}, drawn from {n_sdss} SDSS spectra with {replace=}")
print(f"masked pixel fraction: {1 - mask_appl.mean():.3f}, nan fraction in output: {np.isnan(noisy_flux_sim).mean():.3f}")

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(10, 6), sharex=True, layout="none")
i = 6
x_range = (3800, 4500)

ax[0].plot(wave_sim, sigma_F_appl[i], color="orange", label="SDSS noise")
ax[0].tick_params(labelbottom=False)
ax[0].set_ylim(0, np.nanmax(sigma_F_appl[i])+np.nanmax(sigma_F_appl[i])*0.1)
ax[0].set_ylabel(r"$\sigma_F$")
ax[0].legend(fontsize=10)

ax[1].plot(wave_sdss, flux_sdss_appl[i], color="purple", label="SDSS spectrum")
ax[1].set_ylabel("Flux")
ax[1].tick_params(labelbottom=False)
ax[1].set_ylim(0, max(1.99, np.nanmax(flux_sdss_appl[i])))
ax[1].legend(fontsize=10)

ax[2].plot(wave_sim, noisy_flux_sim[i], color="green", label="Simulated + SDSS noise + mask")
ax[2].plot(wave_sim, flux_sim[i], linestyle="--", color="black", label="Simulated", alpha=0.5)
ax[2].set_ylabel("Flux")
ax[2].set_xlabel("Wavelength [$\\AA$]")
ax[2].set_ylim(0, max(1.99, np.nanmax(noisy_flux_sim[i])))
ax[2].legend(fontsize=10)

fig.tight_layout()
fig.subplots_adjust(hspace=0)

fig.savefig("plots/noisy_spectra_comp.pdf", format="pdf", dpi=300)